# Neural Network SMS Text Classifier

Binary classifier (ham / spam) built with a TextVectorization layer, an embedding and a bidirectional LSTM.

In [1]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
tf.random.set_seed(42)
np.random.seed(42)
print(tf.__version__)

2.16.1


In [2]:
# get data files
if os.path.exists('data/train-data.tsv'):
    train_file_path = 'data/train-data.tsv'
    test_file_path = 'data/valid-data.tsv'
else:
    if not os.path.exists('train-data.tsv'):
        !wget -q https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
        !wget -q https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv
    train_file_path = 'train-data.tsv'
    test_file_path = 'valid-data.tsv'

In [3]:
train_data = pd.read_csv(train_file_path, sep='\t', names=['label', 'message'])
valid_data = pd.read_csv(test_file_path, sep='\t', names=['label', 'message'])

train_text = train_data['message'].values
valid_text = valid_data['message'].values
train_labels = (train_data['label'].values == 'spam').astype('float32')
valid_labels = (valid_data['label'].values == 'spam').astype('float32')
print(train_data.shape, valid_data.shape)

(4179, 2) (1392, 2)


In [4]:
max_features = 10000
sequence_length = 120

vectorizer = layers.TextVectorization(
    max_tokens=max_features, output_mode='int',
    output_sequence_length=sequence_length)
vectorizer.adapt(train_text)

model = keras.Sequential([
    vectorizer,
    layers.Embedding(max_features, 32),
    layers.Bidirectional(layers.LSTM(32)),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid'),
])
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='binary_crossentropy', metrics=['accuracy'])
history = model.fit(train_text, train_labels, epochs=12, batch_size=64,
                    validation_data=(valid_text, valid_labels), verbose=2)
print('validation accuracy:', history.history['val_accuracy'][-1])

Epoch 1/12


66/66 - 50s - 764ms/step - accuracy: 0.8689 - loss: 0.3749 - val_accuracy: 0.9519 - val_loss: 0.1855


Epoch 2/12


66/66 - 31s - 470ms/step - accuracy: 0.9734 - loss: 0.1052 - val_accuracy: 0.9648 - val_loss: 0.1015


Epoch 3/12


66/66 - 27s - 413ms/step - accuracy: 0.9902 - loss: 0.0383 - val_accuracy: 0.9763 - val_loss: 0.0657


... (15 outputs left out)

66/66 - 30s - 454ms/step - accuracy: 0.9995 - loss: 9.0642e-04 - val_accuracy: 0.9849 - val_loss: 0.0843


Epoch 12/12


66/66 - 28s - 431ms/step - accuracy: 1.0000 - loss: 5.3901e-04 - val_accuracy: 0.9835 - val_loss: 0.0899


validation accuracy: 0.983476996421814


In [5]:
# function to predict messages based on model
# (should return list containing prediction and label, ex. [0.008318834938108921, 'ham'])
def predict_message(pred_text):
    batch = tf.constant([pred_text])
    probability = float(model.predict(batch, verbose=0)[0][0])
    return [probability, 'spam' if probability > 0.5 else 'ham']


pred_text = 'how are you doing today?'
prediction = predict_message(pred_text)
print(prediction)

[4.080011785845272e-05, 'ham']


In [6]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
    test_messages = ['how are you doing today',
                     'sale today! to stop texts call 98912460324',
                     'i dont want to go. can we try it a different day? available sat',
                     'our new mobile video service is live. just install on your phone to start watching.',
                     'you have won £1000 cash! call to claim your prize.',
                     "i'll bring it tomorrow. don't forget the milk.",
                     'wow, is your arm alright. that happened to me one time too']

    test_answers = ['ham', 'spam', 'ham', 'spam', 'spam', 'ham', 'ham']
    passed = True

    for msg, ans in zip(test_messages, test_answers):
        prediction = predict_message(msg)
        if prediction[1] != ans:
            passed = False

    if passed:
        print('You passed the challenge. Great job!')
    else:
        print("You haven't passed yet. Keep trying.")

test_predictions()

You passed the challenge. Great job!
